# Local Green, Global Red III: 06 Reviewer Verification
## What is proved, what is computed, and what remains conditional?

**Author:** Jeremy H. Carroll  
**Date:** 2026-10-15  
**Version:** 1.5.2  

---

### 1. Verification Purpose and Claims Matrix

This notebook provides the formal verification bridge for reviewers. It ties the paper's mathematical claims (ClaimIR rows LGGR3-C01 through LGGR3-C15) directly to their mechanization classes, Lean handles, and production Python tests.

In [ ]:
claims = [
    ('LGGR3-C01', 'Binary evaluator defect definition', 'LEAN_FULL', 'LGGR3.defect'),
    ('LGGR3-C02', 'Defect trichotomy in {-1, 0, 1}', 'LEAN_FULL', 'LGGR3.defect_range'),
    ('LGGR3-C03', 'Local-green/global-red witness', 'LEAN_FULL', 'LGGR3.local_green_global_red'),
    ('LGGR3-C04', 'Global minus local defect sum identity', 'LEAN_FULL', 'LGGR3.global_sub_local_eq_defect_sum'),
    ('LGGR3-C05', 'Pointwise defect cancellation', 'LEAN_FULL', 'LGGR3.binary_defect_cancellation'),
    ('LGGR3-C06', 'Archived phone test totals', 'LEAN_FULL', 'LGGR3.phone_totals'),
    ('LGGR3-C07', 'Integer kernel reindexing', 'LEAN_FULL', 'LGGR3.factorized_integer_kernel_has_zero_residual'),
    ('LGGR3-C08', 'Quotient evaluator constancy', 'LEAN_FULL', 'LGGR3.quotient_evaluator_constant'),
    ('LGGR3-C09', 'Quotient descent obstruction', 'LEAN_FULL', 'LGGR3.quotient_evaluator_obstruction'),
    ('LGGR3-C10', 'Six-role overlay specification', 'LEAN_PARTIAL', 'LGGR3.six_roles'),
    ('LGGR3-C11', 'Observer score weight sensitivity', 'LEAN_FULL', 'LGGR3.observer_weights_can_change_a_score'),
    ('LGGR3-C12', 'Fixed observation preserves defect', 'LEAN_FULL', 'LGGR3.fixed_observation_preserves_defect'),
    ('LGGR3-C13', 'Formal scope boundary', 'SCOPE_FENCE', 'Formal scope boundary'),
    ('LGGR3-C14', 'Constant-output adequacy open bound', 'PYTHON_INSTANCE', 'code/epk_gtla_overlay_core.py'),
    ('LGGR3-C15', 'Foundational citations', 'LITERATURE', 'Foundational citations'),
]

print(f"{'Claim ID':^10} | {'Title':^40} | {'Mechanization':^16} | {'Handle / Citation'}")
print('-' * 95)
for cid, title, mech, handle in claims:
    print(f"{cid:^10} | {title:<40} | {mech:^16} | {handle}")

### 2. Differential Testing: Standalone Formula vs. Production Core

We differential-test the elementary defect formula used across the standalone teaching notebooks against the production `epk_gtla_overlay_core.EvaluatorVerdictTriple` across all eight binary triples.

In [ ]:
import epk_gtla_overlay_core as core

def standalone_defect(e1: int, e2: int, eg: int) -> int:
    return eg - (e1 * e2)

all_match = True
for e1 in (0, 1):
    for e2 in (0, 1):
        for eg in (0, 1):
            d_standalone = standalone_defect(e1, e2, eg)
            prod_triple = core.EvaluatorVerdictTriple(e1, e2, eg)
            d_prod = prod_triple.evaluator_defect
            if d_standalone != d_prod:
                all_match = False
                print(f'MISMATCH at ({e1}, {e2}, {eg}): standalone={d_standalone}, prod={d_prod}')

assert all_match
print('Differential test passed: Standalone formula is bitwise identical to production core across all 8 triples.')

### 3. Production Multiplicity Audit Verification

We verify that the production G1 audit rejects duplicated composite coverage.

In [ ]:
first = (core.PathSegment('ab', 'a', 'b', 1, 1),)
second = (core.PathSegment('bc', 'b', 'c', 2, 1),)
joined_dup = (
    core.CompositePath('abc-1', 'ab', 'bc', 3, 1),
    core.CompositePath('abc-2', 'ab', 'bc', 3, 1),
)
report = core.boundary_bijection_report(first, second, joined_dup)
assert not report.is_bijection
assert any('extra or duplicated' in issue for issue in report.issues)
print('Production G1 audit successfully caught composite duplication:')
print(f'  Issues: {report.issues}')

### 4. Production G3 Cancelling Discrepancy Verification

We verify that production `audit_kernel_composition` computes hypotheses and identity independently.

In [ ]:
p1 = (core.PathSegment('ab', 'a', 'b', 0, 1), core.PathSegment('ad', 'a', 'd', 0, 0))
p2 = (core.PathSegment('bc', 'b', 'c', 0, 1), core.PathSegment('dc', 'd', 'c', 0, 1))
pj = (core.CompositePath('abc', 'ab', 'bc', 0, 0), core.CompositePath('adc', 'ad', 'dc', 0, 1))

audit = core.audit_kernel_composition(p1, p2, pj, decay=0)
assert not audit.hypotheses_hold
assert not audit.g3_evaluator_factorized
assert audit.identity_holds
assert audit.residual == 0.0
print('Production audit correctly confirmed G3 failure alongside zero residual:')
print(f'  Hypotheses hold: {audit.hypotheses_hold}')
print(f'  Residual:        {audit.residual}')

### 5. Reviewer Exercise and Worked Verification

**Exercise:**  
Review Claim LGGR3-C05 in the matrix above (and the numerical kernel audit in Section 4). Explain why an automated test suite that only checks `assert audit.residual == 0.0` is insufficient for verifying compositional correctness.

---

**Worked Answer:**  
As demonstrated in Section 4, opposing errors can cancel each other out in the kernel convolution sum, producing an exact residual of 0.0 even when the underlying evaluation failed to factor (Hypothesis G3 violated). A sound verification test must assert structural hypotheses (`audit.hypotheses_hold`) independently from numerical residuals (`audit.identity_holds`).

---

### 6. Reviewer Summary and Evidence Boundary

**What is proved:**  
- Signed evaluator defect trichotomy $\Delta_E \in \{-1, 0, 1\}$ is proved in Lean 4 (`LGGR3.defect_range`, axiom-free).  
- Finite sum identity $\sum e_g - \sum e_1 e_2 = \sum \Delta_E$ is proved in Lean 4 (`LGGR3.global_sub_local_eq_defect_sum`, using `propext`, `Quot.sound`).  
- Pointwise defect cancellation is proved in Lean 4 (`LGGR3.binary_defect_cancellation`, axiom-free).  
- Quotient descent constancy and obstruction are proved in Lean 4 (`LGGR3.quotient_evaluator_constant`, `LGGR3.quotient_evaluator_obstruction`, using `Quot.sound`).  
- Observer scoring sensitivity and defect preservation are proved in Lean 4 (`LGGR3.observer_weights_can_change_a_score`, `LGGR3.fixed_observation_preserves_defect`, axiom-free).  
- Pairwise bijection multiplicity requirements are audited by production Python core.

**What is computed:**  
- Pointwise defects and sample summaries.  
- Kernel composition residuals and boundary bijection checks.

**What remains conditional:**  
- The composition theorem is conditional on G1–G4. Accidental zero residuals when G3 fails do not satisfy the theorem.

---

**Evidence Boundary:**  
This package supplies finite mathematical tools and formal proofs for localized evaluator defects. It does not certify universal reliability, end-to-end hardware correctness, or general AI alignment.